# 02 — Train M0 (baseline) then M1 (curriculum)
M0 must finish first: M1's config asserts against M0's `run_meta.json` (checkpoint hash + step count). Debug on the 0.5B config before spending T4/L4 hours on 1.5B/3B (Part 2 practical plan).

In [ ]:

# --- Self-contained Colab bootstrap (Part 0) ---
# Every notebook does this independently: Colab does not guarantee a new
# notebook tab reuses a previous notebook's VM, so nothing installed or
# cloned in another notebook can be assumed to exist here. This is
# idempotent -- re-running it (e.g. because you ARE still on the same
# runtime) just no-ops the clone and re-pulls latest.
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')

REPO = "https://github.com/saiswaroop25-pixel/verilog-slm"
if not os.path.exists('/content/verilog-slm'):
    !git clone {REPO} /content/verilog-slm
%cd /content/verilog-slm
!git pull

CKPT = '/content/drive/MyDrive/verilog-slm/checkpoints'
LOGS = '/content/drive/MyDrive/verilog-slm/logs'
os.makedirs(CKPT, exist_ok=True); os.makedirs(LOGS, exist_ok=True)
!ln -sfn {CKPT} artifacts_drive_ckpt
!ln -sfn {LOGS} artifacts_drive_logs

In [ ]:
# Pinned deps (Part 0 hygiene: Colab silently upgrades packages between sessions)
!pip install -q -r requirements.txt -r requirements-train.txt

In [ ]:
# RTL toolchain: iverilog is required (compile+simulate). yosys and verible
# are optional -- the harness soft-gates on them (see docs/industry_standards.md)
# but install them here so the synthesis/lint stages actually run instead of
# being recorded as 'skipped'.
!apt-get -qq update && apt-get -qq install -y iverilog yosys > /dev/null
!curl -sL https://github.com/chipsalliance/verible/releases/latest/download/verible-linux-static-x86_64.tar.gz -o /tmp/verible.tar.gz
!mkdir -p /opt/verible && tar -xzf /tmp/verible.tar.gz -C /opt/verible --strip-components=1
os.environ['PATH'] += ':/opt/verible/bin'
!iverilog -V | head -1
!yosys -V
!verible-verilog-lint --version

In [ ]:
# Record the GPU model at the start of every run -- required for the
# per-GPU-hour metric (Part 0 non-negotiable hygiene) to mean anything.
!nvidia-smi -L

## Debug pass (0.5B) -- minutes, not hours
Swap `model.name` to `Qwen/Qwen2.5-Coder-0.5B` in a scratch config first and confirm the loop runs end to end before touching the real budget.

In [ ]:
!python - <<'PY'
import yaml
cfg = yaml.safe_load(open('configs/m0_baseline.yaml'))
cfg['extends'] = 'base.yaml'
with open('configs/_debug_m0.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)
PY
# then hand-edit configs/base.yaml's model.name to the 0.5B checkpoint
# temporarily, or pass an override -- kept manual and explicit here so a
# debug run can never accidentally become the real M0.

## M0 -- flat SFT baseline

In [ ]:
!python -m src.train.sft --config configs/m0_baseline.yaml 2>&1 | tee artifacts_drive_logs/m0_stdout.log
!cp -r artifacts/m0 artifacts_drive_ckpt/m0

## Diagnostic pass (Part 6) -- produces the table M1's reweighting needs
Run before M1. See notebooks/03_diagnose.ipynb for the full breakdown; the minimum needed here is `artifacts/m0_diagnostic.json`.

In [ ]:
!python -m src.infer.generate --adapter artifacts/m0/final --split probe \
  --n 5 --temperature 0.8 --top_p 0.95 --out artifacts/m0_probe_gens.jsonl
!python -m src.eval.diagnose --gens artifacts/m0_probe_gens.jsonl --out artifacts/m0_diagnostic.json

## M1 -- curriculum SFT, same step budget as M0
`assert_matches_m0` inside sft.py fails loudly if the base checkpoint or step count diverge from M0 -- this is the guarantee that keeps the ablation clean (Part 8).

In [ ]:
!python -m src.train.sft --config configs/m1_curriculum.yaml 2>&1 | tee artifacts_drive_logs/m1_stdout.log
!cp -r artifacts/m1 artifacts_drive_ckpt/m1

In [ ]:
# Plot realised category histogram: M1 actually saw vs. M0 (uniform) --
# the direct evidence the curriculum did what it was designed to do (Part 7).
import json, matplotlib.pyplot as plt
m1_meta = json.load(open('artifacts/m1/run_meta.json'))
hist = m1_meta['realised_histogram']['construct']
plt.bar(hist.keys(), hist.values())
plt.xticks(rotation=60, ha='right')
plt.title('M1 realised construct-tag exposure over training')
plt.tight_layout()
plt.savefig('artifacts/m1_realised_histogram.png')
plt.show()